In [ ]:
# Install Open Babel
!conda install -c conda-forge openbabel
!conda install python=3.11.0 -c conda-forge -y
# Install AutoDock Vina
!conda install -c bioconda autodock-vina

In [ ]:
# Check Open Babel version
!obabel -V

# Check AutoDock Vina version
!vina --version

In [ ]:
!pip install transformers rdkit pandas torch

In [ ]:
# Install required libraries
!pip install transformers rdkit pandas torch
!conda install -c conda-forge openbabel
!conda install -c bioconda autodock-vina

import numpy as np
import torch
from transformers import EsmModel, EsmTokenizer
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
import subprocess
import os

# Load the ESM-3 model and tokenizer
model_name = 'facebook/esm3'
model = EsmModel.from_pretrained(model_name)
tokenizer = EsmTokenizer.from_pretrained(model_name)

def get_protein_embeddings(sequence):
    inputs = tokenizer(sequence, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state
    return embeddings.mean(dim=1).numpy()

# Example protein sequence for FTD
protein_sequence = "M A D S T R O N I C S T I C I T I O N S C H M I F N N A T K A F K S S"
protein_embeddings = get_protein_embeddings(protein_sequence)
print(f"Protein Embeddings: {protein_embeddings}")

# Generate SMILES strings for ligands
def generate_smiles(num_samples):
    smiles_list = [
        'CCO',  # Ethanol
        'CCN',  # Ethylamine
        'CCCN', # Propylamine
        'CCOCC',# Diethyl ether
        'CCOCCO' # Diethylene glycol
    ]
    return smiles_list[:num_samples]

new_smiles = generate_smiles(5)
print(f"Generated SMILES: {new_smiles}")

# Predict Binding Affinities
def extract_descriptors(mol):
    if mol is None:
        return np.zeros(5)
    descriptors = [
        Descriptors.MolWt(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumRotatableBonds(mol),
    ]
    return np.array(descriptors)

def predict_binding_affinity(smiles):
    mol = Chem.MolFromSmiles(smiles)
    descriptors = extract_descriptors(mol)
    return np.sum(descriptors)

predicted_affinities = [predict_binding_affinity(smiles) for smiles in new_smiles]
print(f"Predicted Binding Affinities: {predicted_affinities}")

# Molecular Docking
def prepare_pdbqt_file(smiles, output_file):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol)
    pdb_file = 'temp.pdb'
    pdbqt_file = output_file
    with open(pdb_file, 'w') as f:
        f.write(Chem.MolToPDBBlock(mol))
    subprocess.run(['obabel', pdb_file, '-O', pdbqt_file, '--add-hydrogens', '--convert', 'pdbqt'])
    return pdbqt_file

def run_docking(protein_pdbqt, ligand_pdbqt, output_file):
    config_file = 'config.txt'
    with open(config_file, 'w') as f:
        f.write(f'receptor = {protein_pdbqt}\n')
        f.write(f'ligand = {ligand_pdbqt}\n')
        f.write('center_x = 0\n')
        f.write('center_y = 0\n')
        f.write('center_z = 0\n')
        f.write('size_x = 20\n')
        f.write('size_y = 20\n')
        f.write('size_z = 20\n')

    subprocess.run(['vina', '--config', config_file, '--out', output_file])

# Example protein PDBQT file path (make sure this file exists)
protein_pdbqt = 'ftd_protein.pdbqt'  # Replace with actual FTD protein file
docking_results_dir = 'docking_results/'
os.makedirs(docking_results_dir, exist_ok=True)

for i, smiles in enumerate(new_smiles):
    ligand_pdbqt = prepare_pdbqt_file(smiles, f'ligand_{i}.pdbqt')
    run_docking(protein_pdbqt, ligand_pdbqt, f'{docking_results_dir}/docking_result_{i}.pdbqt')

print("Docking completed. Check the results in the 'docking_results/' directory.")


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors

def generate_smiles(num_samples):
    smiles_list = [
        'CCO',  # Ethanol
        'CCN',  # Ethylamine
        'CCCN', # Propylamine
        'CCOCC',# Diethyl ether
        'CCOCCO' # Diethylene glycol
    ]
    return smiles_list[:num_samples]

new_smiles = generate_smiles(5)
print(f"Generated SMILES: {new_smiles}")

In [ ]:
def extract_descriptors(mol):
    if mol is None:
        return np.zeros(5)  # Return a zero vector if the molecule is invalid
    descriptors = [
        Descriptors.MolWt(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumRotatableBonds(mol),
    ]
    return np.array(descriptors)

# Dummy binding affinity model (replace with actual model)
def predict_binding_affinity(smiles):
    mol = Chem.MolFromSmiles(smiles)
    descriptors = extract_descriptors(mol)
    # Dummy prediction: Just sum the descriptors for demonstration
    return np.sum(descriptors)

predicted_affinities = [predict_binding_affinity(smiles) for smiles in new_smiles]
print(f"Predicted Binding Affinities: {predicted_affinities}")


In [ ]:
import subprocess
import os

def prepare_pdbqt_file(smiles, output_file):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol)
    pdb_file = 'temp.pdb'
    pdbqt_file = output_file
    with open(pdb_file, 'w') as f:
        f.write(Chem.MolToPDBBlock(mol))
    subprocess.run(['obabel', pdb_file, '-O', pdbqt_file, '--add-hydrogens', '--convert', 'pdbqt'])
    return pdbqt_file

def run_docking(protein_pdbqt, ligand_pdbqt, output_file):
    config_file = 'config.txt'
    with open(config_file, 'w') as f:
        f.write(f'receptor = {protein_pdbqt}\n')
        f.write(f'ligand = {ligand_pdbqt}\n')
        f.write('center_x = 0\n')
        f.write('center_y = 0\n')
        f.write('center_z = 0\n')
        f.write('size_x = 20\n')
        f.write('size_y = 20\n')
        f.write('size_z = 20\n')

    subprocess.run(['vina', '--config', config_file, '--out', output_file])

# Example protein PDBQT file path (make sure this file exists)
protein_pdbqt = 'huntingtin_protein.pdbqt'
docking_results_dir = 'docking_results/'
os.makedirs(docking_results_dir, exist_ok=True)

for i, smiles in enumerate(new_smiles):
    ligand_pdbqt = prepare_pdbqt_file(smiles, f'ligand_{i}.pdbqt')
    run_docking(protein_pdbqt, ligand_pdbqt, f'{docking_results_dir}/docking_result_{i}.pdbqt')

print("Docking completed. Check the results in the 'docking_results/' directory.")
